# Cybersecurity Intrusion Detection Using Machine Learning

**Course:** ECSCI24302 (Problem-Based Learning) &middot; **Dataset:** UNSW-NB15 &middot; Binary + optional multi-class

This notebook is the worked end-to-end pipeline: dataset verification, data-quality
audit, exploratory data analysis, leaks-safe preprocessing, baseline + four ML models,
hyper-parameter tuning, threshold selection, a **single** pass over the untouched
testing partition, comparison, error analysis, interpretation, robustness checks, an
optional multi-class extension, and a demonstration on genuine held-out records.

> All numbers quoted here are produced by executing the cells below and are therefore
> identical to `results/findings.md`, which is the single source of truth for the
> report, the slides and the viva.

**Leakage rules enforced throughout:**
1. Data-dependent preprocessing is fitted on training data only, *after* splitting.
2. The released testing partition is never cleaned, never used for tuning, model
   selection or threshold decisions - it is evaluated exactly once at the end.
3. `id`, `label` and `attack_cat` are excluded from model inputs (no target leakage);
   no target-derived predictors are used anywhere.
4. Every model is wrapped in the same `Pipeline` to guarantee identical processing.

## 0. Environment and project paths

In [1]:
import sys
from pathlib import Path

# Works locally (walks up to the folder containing src/train.py)
project_root = Path().resolve()
while not (project_root / "src" / "train.py").exists() and project_root != project_root.parent:
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)


Project root: C:\Users\Rudra Patel\college\ML Project\Cybersecurity_Intrusion_Detection


In [2]:
%matplotlib inline
import warnings
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
print("Run started:", datetime.now().strftime("%Y-%m-%d %H:%M"))

Run started: 2026-10-26 22:02


## 1. Dataset provenance and verified loading

| Item | Value |
|---|---|
| Name | **UNSW-NB15** - official partitioned release (UNSW Canberra Cyber, 2015) |
| Access page | https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15 |
| Design | Synthetic hybrid of real captures; 10 attack families; 49 flow features + 2 annotations |
| Files used | `UNSW_NB15_training.csv` (175,341 rows) and `UNSW_NB15_testing.csv` (82,332 rows), 45 columns each |
| Split discipline | The official **training** partition is used for fitting/tuning; the official **testing** partition stays untouched until the final evaluation |

`load_and_validate` re-derives every dataset fact (row counts, columns, id ranges,
missing cells, label consistency) instead of trusting any hard-coded claim, and raises
if the release does not match the official spec.

In [3]:
from src.data_loading import (DATASET_INFO, EXPECTED_COLUMNS,
                              FEATURES_EXCLUDED_FROM_MODELING,
                              dataset_summary_table, load_and_validate)
from src.preprocessing import audit_quality

print("Excluded from modeling:", FEATURES_EXCLUDED_FROM_MODELING)
train_df, test_df, facts = load_and_validate()
facts

Excluded from modeling: ['id', 'label', 'attack_cat']


{'train': {'rows': 175341,
  'columns': 45,
  'unexpected_extra_columns': [],
  'label_counts': {1: 119341, 0: 56000},
  'benign_pct': 31.938,
  'malicious_pct': 68.062,
  'attack_cat_values': ['Analysis',
   'Backdoor',
   'DoS',
   'Exploits',
   'Fuzzers',
   'Generic',
   'Normal',
   'Reconnaissance',
   'Shellcode',
   'Worms'],
  'label_attack_cat_mismatches': 0,
  'duplicate_rows': 67601,
  'id_range': [1, 175341]},
 'test': {'rows': 82332,
  'columns': 45,
  'unexpected_extra_columns': [],
  'label_counts': {1: 45332, 0: 37000},
  'benign_pct': 44.94,
  'malicious_pct': 55.06,
  'attack_cat_values': ['Analysis',
   'Backdoor',
   'DoS',
   'Exploits',
   'Fuzzers',
   'Generic',
   'Normal',
   'Reconnaissance',
   'Shellcode',
   'Worms'],
  'label_attack_cat_mismatches': 0,
  'duplicate_rows': 26387,
  'id_range': [1, 82332]}}

## 2. Dataset facts

In [4]:
import json
print("TRAIN:", json.dumps(facts["train"], indent=2))
print("TEST :", json.dumps(facts["test"], indent=2))

TRAIN: {
  "rows": 175341,
  "columns": 45,
  "unexpected_extra_columns": [],
  "label_counts": {
    "1": 119341,
    "0": 56000
  },
  "benign_pct": 31.938,
  "malicious_pct": 68.062,
  "attack_cat_values": [
    "Analysis",
    "Backdoor",
    "DoS",
    "Exploits",
    "Fuzzers",
    "Generic",
    "Normal",
    "Reconnaissance",
    "Shellcode",
    "Worms"
  ],
  "label_attack_cat_mismatches": 0,
  "duplicate_rows": 67601,
  "id_range": [
    1,
    175341
  ]
}
TEST : {
  "rows": 82332,
  "columns": 45,
  "unexpected_extra_columns": [],
  "label_counts": {
    "1": 45332,
    "0": 37000
  },
  "benign_pct": 44.94,
  "malicious_pct": 55.06,
  "attack_cat_values": [
    "Analysis",
    "Backdoor",
    "DoS",
    "Exploits",
    "Fuzzers",
    "Generic",
    "Normal",
    "Reconnaissance",
    "Shellcode",
    "Worms"
  ],
  "label_attack_cat_mismatches": 0,
  "duplicate_rows": 26387,
  "id_range": [
    1,
    82332
  ]
}


In [5]:
summary_df = dataset_summary_table(train_df, test_df)
summary_df

,column,dtype_train,non_missing_train,non_missing_test,unique_train,unique_test,role
0,id,int64,175341,82332,175341,82332,identifier
1,dur,float64,175341,82332,74039,39888,input feature
2,proto,str,175341,82332,133,131,input feature
3,service,str,175341,82332,13,13,input feature
4,state,str,175341,82332,9,7,input feature
5,spkts,int64,175341,82332,480,420,input feature
6,dpkts,int64,175341,82332,443,436,input feature
7,sbytes,int64,175341,82332,7214,4489,input feature
8,dbytes,int64,175341,82332,6660,4034,input feature
9,rate,float64,175341,82332,76991,40616,input feature


## 3. Data-quality audit

`audit_quality` inventories exactly-duplicate flows, constant columns, category
cardinality, unseen test categories and label/target consistency. It also fixes the
`drop_constant` exclusion list; decisions come from training data only - the testing
partition is never cleaned.

In [6]:
audit = audit_quality(train_df, test_df)

drop_constant = [c for c, n_uniq in audit["train"]["constant_columns"].items()
                 if c not in FEATURES_EXCLUDED_FROM_MODELING and n_uniq <= 1]
print("Constant feature columns found:", drop_constant)
print("Exact duplicate rows  - training:", audit["train"]["duplicate_rows_excluding_id"],
      "| testing:", audit["test"]["duplicate_rows_excluding_id"])
print("Distinct flow tuples shared with the testing file:",
      audit["distinct_flow_tuples_shared_with_test"])

Constant feature columns found: []
Exact duplicate rows  - training: 67601 | testing: 26387
Distinct flow tuples shared with the testing file: 940


## 4. Exploratory data analysis (figures 1-7)

`run_eda` regenerates figures 1-7 and `results/eda_summary.json` (class balance,
box-plot separations, correlations with the target, categorical composition,
duplicates and missingness).

In [7]:
from src import eda as eda_mod

eda_summary = eda_mod.run_eda(train_df, test_df, audit, summary_df)
print("EDA figures saved to results/figures/. Top correlations with 'malicious':")
pd.Series(eda_summary["correlation"]["top_positive_with_label"]).head(8)

EDA figures saved to results/figures/. Top correlations with 'malicious':


sttl                0.692741
ct_state_ttl        0.577704
ct_dst_sport_ltm    0.357213
rate                0.337979
ct_src_dport_ltm    0.305579
ct_dst_src_ltm      0.303855
ct_src_ltm          0.238225
ct_dst_ltm          0.229887
dtype: float64

## 5. Cleaning the TRAINING partition (deduplication only)

In [8]:
cols = [c for c in train_df.columns if c != "id"]
n_before = len(train_df)
train_dedup = train_df.drop_duplicates(subset=cols, keep="first")
print(f"Training deduplication: {n_before:,} -> {len(train_dedup):,} rows")
print(f"Testing partition rows kept exactly as released: {len(test_df):,}")

Training deduplication: 175,341 -> 107,740 rows
Testing partition rows kept exactly as released: 82,332


## 6. Identical pipelines for every model

Feature engineering + preprocessing live inside the pipeline, so they are always fitted
on training data only and never leak.

In [9]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from src.preprocessing import FeatureEngineer, build_preprocessor

def wrap(model, scale=False):
    return Pipeline([
        ("feature_engineering", FeatureEngineer()),
        ("preprocess", build_preprocessor(drop_constant, scale=scale)),
        ("model", model),
    ])

models = {
    "Dummy": wrap(DummyClassifier(strategy="most_frequent")),
    "LogisticRegression": wrap(LogisticRegression(C=1.0, max_iter=3000, class_weight="balanced", solver="lbfgs", random_state=RANDOM_STATE), scale=True),
    "DecisionTree": wrap(DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=RANDOM_STATE)),
    "RandomForest": wrap(RandomForestClassifier(n_estimators=200, class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE)),
    "HistGradientBoosting": wrap(HistGradientBoostingClassifier(max_iter=250, learning_rate=0.1, l2_regularization=0.0, random_state=RANDOM_STATE)),
}
list(models)

['Dummy',
 'LogisticRegression',
 'DecisionTree',
 'RandomForest',
 'HistGradientBoosting']

## 7. Baseline on an internal validation slice

The dummy baseline is fitted on a 80/20 internal split of the training partition.
The test partition is never touched here.

In [10]:
from sklearn.model_selection import train_test_split
from src.evaluate import classification_metrics

y_all = train_dedup["label"]
train_fit, valid_internal = train_test_split(train_dedup, test_size=0.20,
                                             stratify=y_all, random_state=RANDOM_STATE)

baseline = models["Dummy"]
baseline.fit(train_fit, train_fit["label"])
proba_b = baseline.predict_proba(valid_internal)[:, 1]
pred_b = (proba_b >= 0.5).astype(int)
m_b = classification_metrics(valid_internal["label"].to_numpy(), pred_b, y_proba=proba_b)
{"model": "Dummy (most_frequent)", **{k: round(float(m_b[k]), 4)
     for k in ("accuracy", "precision", "recall", "f1", "roc_auc")}}

{'model': 'Dummy (most_frequent)',
 'accuracy': 0.5184,
 'precision': 0.5184,
 'recall': 1.0,
 'f1': 0.6828,
 'roc_auc': 0.5}

## 8. Hyper-parameter tuning (training partition only, cv=3, scoring=F1)

Search spaces are fixed a priori and documented in `results/tuning_*.json`;
GridSearchCV for LogisticRegression/DecisionTree, RandomizedSearchCV for
RandomForest/HistGradientBoosting, on a stratified 60,000-row subsample.

In [11]:
import time as _t
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from src.train import SEARCH_SPACES, MAX_SEARCH_ROWS, SEARCH_SPLITS
RS = RANDOM_STATE

Xs = train_dedup
ys = Xs["label"]
if len(Xs) > MAX_SEARCH_ROWS:
    Xs, _, ys, _ = train_test_split(Xs, ys, train_size=MAX_SEARCH_ROWS,
                                    stratify=ys, random_state=RS)
print(f"Tuning subsample: {len(Xs):,} stratified rows")

tuned = {}
for name, (searcher_cls, space, n_iter) in SEARCH_SPACES.items():
    t0 = _t.time()
    if searcher_cls is RandomizedSearchCV:
        search = RandomizedSearchCV(clone(models[name]), space, n_iter=n_iter,
                                    cv=SEARCH_SPLITS, scoring="f1", n_jobs=-1,
                                    random_state=RS, refit=True, return_train_score=True)
    else:
        search = GridSearchCV(clone(models[name]), space, cv=SEARCH_SPLITS,
                              scoring="f1", n_jobs=-1, refit=True,
                              return_train_score=True)
    search.fit(Xs, ys)
    tuned[name] = {
        "tuner": searcher_cls.__name__,
        "search_time_s": round(_t.time() - t0, 1),
        "best_params": {k.replace("model__", ""): v for k, v in search.best_params_.items()},
        "best_cv_f1_on_subsample": round(float(search.best_score_), 4),
        "n_candidates": len(search.cv_results_["params"]),
    }
    print(f"{name}: best CV F1 {tuned[name]['best_cv_f1_on_subsample']:.4f} "
          f"in {tuned[name]['search_time_s']}s, best params {tuned[name]['best_params']}")
_tu = tuned
print("Tuning complete for all models.")

Tuning subsample: 60,000 stratified rows


LogisticRegression: best CV F1 0.9119 in 7.8s, best params {'C': 3.0}


DecisionTree: best CV F1 0.9209 in 8.5s, best params {'class_weight': None, 'max_depth': 12, 'min_samples_leaf': 1}


RandomForest: best CV F1 0.9346 in 99.4s, best params {'n_estimators': 400, 'min_samples_leaf': 2, 'max_features': 0.3, 'max_depth': 16, 'class_weight': None}


HistGradientBoosting: best CV F1 0.9359 in 34.1s, best params {'min_samples_leaf': 40, 'max_leaf_nodes': 63, 'max_iter': 300, 'learning_rate': 0.03, 'l2_regularization': 1.0}
Tuning complete for all models.


## 9. Final tuned pipelines + 5-fold validation on the full training partition

In [12]:
final_models = {"Dummy": models["Dummy"]}
for name, info in _tu.items():
    cfg = clone(models[name])
    est = cfg.named_steps["model"]
    params = {f"model__{k}": v for k, v in info["best_params"].items() if k in est.get_params()}
    final_models[f"{name} (tuned)"] = cfg.set_params(**params)

from sklearn.model_selection import StratifiedKFold, cross_validate
y_full = train_dedup["label"].to_numpy()
scoring = {"accuracy": "accuracy", "recall": "recall", "precision": "precision",
           "f1": "f1", "roc_auc": "roc_auc", "average_precision": "average_precision"}
rows = []
for name, model in final_models.items():
    res = cross_validate(model, train_dedup, y_full,
                         cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RS),
                         scoring=scoring, n_jobs=-1)
    rows.append({"model": name,
                 "accuracy": round(float(res["test_accuracy"].mean()), 4),
                 "precision": round(float(res["test_precision"].mean()), 4),
                 "recall": round(float(res["test_recall"].mean()), 4),
                 "f1": round(float(res["test_f1"].mean()), 4),
                 "f1_std": round(float(res["test_f1"].std()), 4),
                 "roc_auc": round(float(res["test_roc_auc"].mean()), 4),
                 "avg_precision": round(float(res["test_average_precision"].mean()), 4),
                 "fit_time_s": round(float(res["fit_time"].mean()), 1)})
cv_table = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)
cv_table

,model,accuracy,precision,recall,f1,f1_std,roc_auc,avg_precision,fit_time_s
0,HistGradientBoosting (tuned),0.9357,0.9243,0.9541,0.9390,0.0013,0.9872,0.9876,38.1
1,RandomForest (tuned),0.9344,0.9175,0.9598,0.9382,0.0011,0.9870,0.9875,63.6
2,DecisionTree (tuned),0.9227,0.9097,0.9448,0.9268,0.0015,0.9788,0.9746,2.7
3,LogisticRegression (tuned),0.9037,0.8623,0.9690,0.9125,0.0009,0.9695,0.9678,7.0
4,Dummy,0.5184,0.5184,1.0000,0.6828,0.0000,0.5000,0.5184,0.8


## 10. Model selection (validation evidence) and operating threshold

Selection rule (fixed a priori): **highest mean CV F1 among tuned candidates; the
dummy baseline is never eligible.** Threshold rule: maximise F1 on a stratified 20%
internal validation slice - never on test.

In [13]:
from src.evaluate import threshold_curve

chosen_name = cv_table.iloc[0]["model"]
print("Selected:", chosen_name)

final_model = final_models[chosen_name]
train_fit2, valid2 = train_test_split(train_dedup, test_size=0.20,
                                      stratify=train_dedup["label"], random_state=RS)
model_val = clone(final_model).fit(train_fit2, train_fit2["label"])
proba_v = model_val.predict_proba(valid2)[:, 1]

grid = np.linspace(0.02, 0.98, 49)
thr_curve = threshold_curve(valid2["label"].to_numpy(), proba_v,
                            "fig12_threshold_tradeoff.png", grid=grid)
threshold = float(thr_curve.loc[thr_curve["f1"].idxmax(), "threshold"])
print(f"F1-optimal threshold on validation = {threshold:.3f}")

Selected: HistGradientBoosting (tuned)


F1-optimal threshold on validation = 0.480


## 11. SINGLE evaluation pass on the untouched testing partition

Threshold and tuning are already fixed. Here - and only here - does the model see
the testing partition.

In [14]:
from src.evaluate import (calibration_plot, comparison_chart, confusion_plot,
                          roc_pr_plots)

y_test = test_df["label"].to_numpy()
probas, mrows, fitted_models = {}, [], {}
for name, model in final_models.items():
    fitted = clone(model).fit(train_dedup, train_dedup["label"])
    fitted_models[name] = fitted
    proba = fitted.predict_proba(test_df)[:, 1]
    pred = (proba >= threshold).astype(int)
    m = classification_metrics(y_test, pred, y_proba=proba, threshold=threshold)
    probas[name] = proba
    mrows.append({"model": name,
                  "accuracy": m["accuracy"], "precision": m["precision"],
                  "recall": m["recall"], "f1": m["f1"],
                  "roc_auc": m["roc_auc"], "avg_precision": m["average_precision"],
                  "FPR": m["false_positive_rate"], "FNR": m["false_negative_rate"]})
final_model = fitted_models[chosen_name]  # now fitted; used for demo / interpretation
test_metrics = pd.DataFrame(mrows).round(4)
test_metrics

,model,accuracy,precision,recall,f1,roc_auc,avg_precision,FPR,FNR
0,Dummy,0.5506,0.5506,1.0000,0.7102,0.5000,0.5506,1.0000,0.0000
1,LogisticRegression (tuned),0.8100,0.7550,0.9696,0.8490,0.9531,0.9650,0.3854,0.0304
2,DecisionTree (tuned),0.8601,0.8098,0.9749,0.8847,0.9693,0.9680,0.2806,0.0251
3,RandomForest (tuned),0.8663,0.8099,0.9894,0.8907,0.9841,0.9881,0.2845,0.0106
4,HistGradientBoosting (tuned),0.8712,0.8154,0.9903,0.8944,0.9868,0.9903,0.2747,0.0097


In [15]:
proba_final = probas[chosen_name]
pred_final_num = (proba_final >= threshold).astype(int)
m_final = classification_metrics(y_test, pred_final_num, y_proba=proba_final,
                                 threshold=threshold)
print(f"Chosen {chosen_name}: tn={m_final['tn']:,} fp={m_final['fp']:,} "
      f"fn={m_final['fn']:,} tp={m_final['tp']:,}")
confusion_plot(y_test, pred_final_num,
               f"Confusion matrix - {chosen_name}\n(test set, threshold {threshold:.2f})",
               "fig10_confusion_matrix_final.png")
confusion_plot(y_test, pred_final_num,
               f"{chosen_name} - class-normalised\n(test, threshold {threshold:.2f})",
               "fig10b_confusion_matrix_normalized.png", normalize=True)
roc_pr_plots(y_test, probas, "fig08_roc_curves.png", "fig09_pr_curves.png")
calibration_plot(y_test, proba_final, chosen_name, "fig14_calibration.png")
comparison_chart(test_metrics, "fig11_model_comparison.png",
                 metric_cols=("recall", "precision", "f1", "roc_auc"))

Chosen HistGradientBoosting (tuned): tn=26,836 fp=10,164 fn=440 tp=44,892


WindowsPath('C:/Users/Rudra Patel/college/ML Project/Cybersecurity_Intrusion_Detection/results/figures/fig11_model_comparison.png')

## 13. Error analysis (figure 15)

In [16]:
is_fp = (y_test == 0) & (pred_final_num == 1)
is_fn = (y_test == 1) & (pred_final_num == 0)

attack_rows = test_df[y_test == 1]
fn_by_cat = attack_rows[is_fn[y_test == 1]].groupby("attack_cat").size()
totals = attack_rows.groupby("attack_cat").size()
fam = pd.DataFrame({"total": totals, "missed": fn_by_cat.reindex(totals.index).fillna(0).astype(int)})
fam["recall_per_family"] = (fam["total"] - fam["missed"]) / fam["total"]
fam = fam.sort_values("recall_per_family")
fam.round(4)

,total,missed,recall_per_family
attack_cat,,,
Fuzzers,6062,372,0.9386
Analysis,677,11,0.9838
Shellcode,378,3,0.9921
Exploits,11132,45,0.9960
DoS,4089,6,0.9985
Reconnaissance,3496,2,0.9994
Generic,18871,1,0.9999
Backdoor,583,0,1.0000
Worms,44,0,1.0000


In [17]:
from src.evaluate import save_fig

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.bar(fam.index, fam["recall_per_family"], color="#c62828", edgecolor="black")
for i, (cat, row) in enumerate(fam.iterrows()):
    ax.annotate(f"{row['recall_per_family']:.2f}\n({int(row['missed']):,}/{int(row['total']):,})",
                (i, row["recall_per_family"] + 0.01), ha="center", fontsize=8)
ax.set_ylim(0.5, 1.06)
ax.set_ylabel("Share of that family detected (recall)")
ax.set_title(f"Attack-family recall of {chosen_name} (test set)")
ax.tick_params(axis="x", rotation=45)
save_fig(fig, "fig15_recall_by_attack_family.png")
print("fig15 saved")

fig15 saved


## 14. Interpretation - permutation importance (figure 13)

Computed on a stratified 10% sample of the TRAINING partition (validation-side lens;
test stays reserved).

In [18]:
from sklearn.inspection import permutation_importance
_, sample = train_test_split(train_dedup, test_size=0.10,
                             stratify=train_dedup["label"], random_state=RS)
imp = permutation_importance(final_model, sample, sample["label"],
                             n_repeats=5, scoring="f1", n_jobs=-1, random_state=RS)

raw = pd.Series(imp.importances_mean, index=sample.columns).sort_values(ascending=False)
grouped = {}
for col_name, val in raw.items():
    if col_name in FEATURES_EXCLUDED_FROM_MODELING:
        continue  # id / label / attack_cat are not model inputs
    feat = col_name  # pipeline gets raw flow columns; report per raw feature
    grouped[feat] = grouped.get(feat, 0.0) + float(val)

fig, ax = plt.subplots(figsize=(8.5, 6.2))
top = sorted(grouped.items(), key=lambda kv: kv[1], reverse=True)[:15]
ax.barh([k for k, _ in top][::-1], [v for _, v in top][::-1], color="#1565c0")
ax.set_xlabel("Mean decrease in validation F1 when the feature is permuted")
ax.set_title(f"Permutation importance - {chosen_name}")
fig.tight_layout()
save_fig(fig, "fig13_permutation_importance.png")
print("Top-5:", top[:5])

Top-5: [('sttl', 0.3747739163893999), ('dbytes', 0.02278451187747239), ('sbytes', 0.020204891154546024), ('ct_srv_dst', 0.01741282669636912), ('ct_dst_src_ltm', 0.01700392511634439)]


## 15. Robustness / leakage sensitivity

The official release shares some exact flows between the training and testing files.
Primary test metrics stand; this check quantifies the inflation.

In [19]:
keys = train_df[cols].drop_duplicates()
merged = test_df[cols].merge(keys, how="left", indicator=True)
unique_mask = (merged["_merge"] == "left_only").to_numpy()
n_shared = int((~unique_mask).sum())

m_all = classification_metrics(y_test, pred_final_num, y_proba=proba_final)
m_u = classification_metrics(y_test[unique_mask], pred_final_num[unique_mask],
                             y_proba=proba_final[unique_mask])
leakage = pd.DataFrame({
    "subset": ["all test rows", f"unique-only ({int(unique_mask.sum()):,} rows)"],
    "accuracy": [m_all["accuracy"], m_u["accuracy"]],
    "precision": [m_all["precision"], m_u["precision"]],
    "recall": [m_all["recall"], m_u["recall"]],
    "f1": [m_all["f1"], m_u["f1"]]}).round(4)
print("Shared exact flows:", n_shared, f"({n_shared/len(test_df):.2%} of test)")
leakage

Shared exact flows: 4247 (5.16% of test)


,subset,accuracy,precision,recall,f1
0,all test rows,0.8712,0.8154,0.9903,0.8944
1,"unique-only (78,085 rows)",0.8679,0.8090,0.9896,0.8902


## 16. OPTIONAL multi-class attack-family classifier (figure 16)

In [20]:
from sklearn.metrics import classification_report, confusion_matrix

y_tr = train_dedup["attack_cat"]
multi_pipe = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocess", build_preprocessor(drop_constant, scale=False)),
    ("model", RandomForestClassifier(n_estimators=150, max_depth=20, min_samples_leaf=2,
                                     class_weight="balanced_subsample", n_jobs=-1,
                                     random_state=RS)),
])
t0 = _t.time()
multi_pipe.fit(train_dedup, y_tr)
print(f"train {(_t.time()-t0):.0f}s")

y_te = test_df["attack_cat"]
pred_mc = multi_pipe.predict(test_df)
labels = sorted(set(y_tr.unique()) | set(y_te.unique()))
rep = classification_report(y_te, pred_mc, labels=labels, output_dict=True, zero_division=0)
cm = confusion_matrix(y_te, pred_mc, labels=labels)
cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(10.5, 8.6))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)), labels, rotation=90, fontsize=9)
ax.set_yticks(range(len(labels)), labels, fontsize=9)
for i in range(cmn.shape[0]):
    for j in range(cmn.shape[1]):
        if cmn[i, j] >= 0.02:
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center",
                    color="white" if cmn[i, j] > 0.6 else "black", fontsize=8)
ax.set_xlabel("Predicted family")
ax.set_ylabel("Actual family")
ax.set_title("Attack-family classifier - row-normalised confusion matrix (test partition)")
fig.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()
save_fig(fig, "fig16_multiclass_confusion.png")
weak = [(k, round(rep[k]["recall"], 4)) for k in labels
        if isinstance(rep.get(k), dict) and rep[k].get("support", 0) > 0
        and rep[k]["recall"] < 0.5]
print(f"multiclass: accuracy {rep['accuracy']:.4f}, "
      f"macro-F1 {rep['macro avg']['f1-score']:.4f}; weak classes {weak}")
joblib.dump(multi_pipe, project_root / "artifacts" / "multiclass_attack_classifier.joblib", compress=3)

train 4s


multiclass: accuracy 0.6814, macro-F1 0.5129; weak classes [('Analysis', 0.3826), ('DoS', 0.124)]


['C:\\Users\\Rudra Patel\\college\\ML Project\\Cybersecurity_Intrusion_Detection\\artifacts\\multiclass_attack_classifier.joblib']

## 17. Demonstration on genuine held-out records

10 records (5 malicious, 5 benign, seeded rng) taken from the testing partition AFTER
the final evaluation - records the model never saw during fitting.

In [21]:
rng = np.random.RandomState(RS)
attack_idx = np.flatnonzero(test_df["label"].to_numpy() == 1)
benign_idx = np.flatnonzero(test_df["label"].to_numpy() == 0)
chosen_rows = np.sort(np.concatenate([
    rng.choice(attack_idx, 5, replace=False),
    rng.choice(benign_idx, 5, replace=False)]))
sample = test_df.iloc[chosen_rows].copy()
proba_demo = final_model.predict_proba(sample)[:, 1]
demo = sample[["id", "attack_cat", "label"]].copy()
demo["predicted_label"] = (proba_demo >= threshold).astype(int)
demo["attack_probability"] = np.round(proba_demo, 4)
demo["correct"] = demo["predicted_label"] == demo["label"]
demo["model"] = chosen_name
demo.round(4).to_csv(project_root / "results" / "demo_predictions.csv", index=False)
print(f"Demo accuracy on {len(demo)} held-out records: {demo['correct'].mean():.2%}")
demo.filter(items=["id", "attack_cat", "label", "predicted_label",
                   "attack_probability", "correct"])

Demo accuracy on 10 held-out records: 80.00%


,id,attack_cat,label,predicted_label,attack_probability,correct
18942,18943,Generic,1,1,0.9992,True
28228,28229,Normal,0,0,0.0001,True
46463,46464,Backdoor,1,1,0.9925,True
47879,47880,Exploits,1,1,0.9976,True
53130,53131,Fuzzers,1,1,0.9971,True
55614,55615,Generic,1,1,0.7242,True
67701,67702,Normal,0,1,0.9055,False
74406,74407,Normal,0,0,0.3996,True
74893,74894,Normal,0,0,0.0040,True
77520,77521,Normal,0,1,0.5889,False


## 18. Honesty checks and limitations summary

Key numbers must match `results/findings.md` exactly.

In [22]:
limitations = {
    "FPR vs FNR": f"FPR of the chosen model is {m_final['false_positive_rate']:.2%} "
                  f"vs FNR {m_final['false_negative_rate']:.2%} - a consequence of the "
                  "UNSW-NB15 official train/test distribution shift and the max-F1 threshold rule (which favours recall).",
    "importance": "sttl dominates permutation importance (0.3748) partly because it is a "
                  "schema artifact of the synthetic capture; treat feature importance as indicative, not causal.",
    "leakage": f"{n_shared:,} exact duplicate flows ({n_shared/len(test_df):.2%}) are shared "
               f"between the released files; on the unique-only subset F1 goes "
               f"{m_all['f1']:.4f} -> {m_u['f1']:.4f}.",
    "multiclass": "Macro-F1 0.5129 for the 10-family classifier; Analysis and DoS recall are genuinely hard.",
    "demo": f"Demo accuracy {demo['correct'].mean():.0%} on {len(demo)} held-out records - small sample, sanity check only.",
}
for k, v in limitations.items():
    print(f"- {k}: {v}")

- FPR vs FNR: FPR of the chosen model is 27.47% vs FNR 0.97% - a consequence of the UNSW-NB15 official train/test distribution shift and the max-F1 threshold rule (which favours recall).
- importance: sttl dominates permutation importance (0.3748) partly because it is a schema artifact of the synthetic capture; treat feature importance as indicative, not causal.
- leakage: 4,247 exact duplicate flows (5.16%) are shared between the released files; on the unique-only subset F1 goes 0.8944 -> 0.8902.
- multiclass: Macro-F1 0.5129 for the 10-family classifier; Analysis and DoS recall are genuinely hard.
- demo: Demo accuracy 80% on 10 held-out records - small sample, sanity check only.


## 19. Conclusion and reproducibility

- The best tuned model is recorded above with its test metrics - numbers are identical
  across the notebook, `results/findings.md`, `model_comparison.csv`, the DOCX report
  and the PPTX slides, because every consumer quotes the same files.
- Artifacts: `artifacts/intrusion_detection_model.joblib` (final binary pipeline),
  `artifacts/multiclass_attack_classifier.joblib` (optional 10-family classifier), plus
  metadata and 16 result figures.
- Deterministic end-to-end with `random_state=42`; the script version is
  `python -m src.train` (about 6.2 minutes on a 20-core machine).
- Data is not redistributed with the repository - run `python -m src.download_data`
  to fetch it (see README for the selected mirror).

In [23]:
kv = {
    "selected model": chosen_name,
    "threshold": round(float(threshold), 3),
    "test accuracy": round(float(m_final["accuracy"]), 4),
    "test precision": round(float(m_final["precision"]), 4),
    "test recall": round(float(m_final["recall"]), 4),
    "test F1": round(float(m_final["f1"]), 4),
    "test ROC-AUC": round(float(m_final["roc_auc"]), 4),
    "avg precision": round(float(m_final["average_precision"]), 4),
    "tn / fp / fn / tp": f"{int(m_final['tn']):,} / {int(m_final['fp']):,} / "
                         f"{int(m_final['fn']):,} / {int(m_final['tp']):,}",
    "python": sys.version.split()[0],
    "executed": datetime.now().strftime("%Y-%m-%d %H:%M"),
}
for k, v in kv.items():
    print(f"{k:>22}: {v}")

        selected model: HistGradientBoosting (tuned)
             threshold: 0.48
         test accuracy: 0.8712
        test precision: 0.8154
           test recall: 0.9903
               test F1: 0.8944
          test ROC-AUC: 0.9868
         avg precision: 0.9903
     tn / fp / fn / tp: 26,836 / 10,164 / 440 / 44,892
                python: 3.12.10
              executed: 2026-10-26 22:08
